# Hallucination Detection - Colab Runner

This notebook sets up the environment and executes the pipeline on Google Colab GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repository from GitHub
!git clone https://github.com/Bright579523/hallucination-detection.git
%cd hallucination-detection

# Install requirements
!pip install -r requirements.txt

### (Returning Users) Restore from Google Drive
If you already ran the detector before, skip Steps 2-5 and just restore your saved CSV files.

In [ ]:
# SKIP THIS if running for the first time!
# Only use when reopening Colab to restore previous results.
!mkdir -p /content/data
!cp /content/drive/MyDrive/hallucination_project/full_detection_results.csv /content/data/
!cp /content/drive/MyDrive/hallucination_project/hard_negative_dataset.csv /content/data/
print("Restored from Google Drive!")

In [ ]:
# Download COCO val2017 directly on Colab (run once)
!mkdir -p /content/data
!wget http://images.cocodataset.org/zips/val2017.zip -O /content/val2017.zip
!wget http://images.cocodataset.org/annotations/annotations_trainval2017.zip -O /content/annotations.zip

# Extract
!unzip -q /content/val2017.zip -d /content/data/
!unzip -q /content/annotations.zip -d /content/data/

# Cleanup zip files
!rm /content/val2017.zip /content/annotations.zip

# (Optional) Save to Google Drive for future use
!mkdir -p /content/drive/MyDrive/hallucination_project/coco/
!cp -r /content/data/val2017 /content/drive/MyDrive/hallucination_project/coco/

In [ ]:
# Step 4: Build the hard-negative dataset from COCO annotations
!python data/build_dataset.py --mode full --data_dir /content/data

## Step 5: OWL-ViT Detection (Person 2)

Run the OWL-ViT model on every image-prompt pair in the dataset.
This step requires GPU and may take 10-20 minutes for 5,000 rows.

In [ ]:
# Run OWL-ViT detection on the full dataset
!python models/detector.py --mode full --data_dir /content/data

### Save Results to Google Drive
Save the CSV so you do not have to re-run the detector next time.

In [ ]:
# Save detection results to Google Drive
!mkdir -p /content/drive/MyDrive/hallucination_project/
!cp /content/data/full_detection_results.csv /content/drive/MyDrive/hallucination_project/
!cp /content/data/hard_negative_dataset.csv /content/drive/MyDrive/hallucination_project/
print("Saved to Google Drive!")

## Step 6: Hallucination Verification (Person 3)

Run both verifiers to classify predictions as genuine or hallucinated.

In [ ]:
# V1: Confidence Threshold Verifier
!python verifier/threshold_verifier.py --data_dir /content/data

# V2: CLIP Cosine Similarity Verifier
!python verifier/clip_verifier.py --data_dir /content/data

## Step 7: Evaluation & Results (Person 4)

Calculate metrics (Precision, Recall, F1, ROC-AUC) and generate plots.

In [ ]:
# Run evaluation on both verifier results
!python evaluation/evaluate.py --data_dir /content/data